In [12]:
# 练习RNN + Embedding
import torch
import torch.nn as nn
import torch.optim as optim

# 字符表
idx2char = ['e','h','l','o']

# 输入序列 ： hello
x_data = [1,0,2,2,3]

# 目标序列 ： ohlol
y_data = [3,1,2,3,2]

# 基本参数
batch_size = 1
seq_len =5

# 字符类别数量
num_class = len(idx2char)

# 将输入字符编号转换为tensor

# 这里和One-hot的区别是
# 用one-hot时 h->[0,1,0,0]
# 所以输入的形状是【seq_len,batch_size,inputs_size】 ->[5,1,4]
# 现在使用Embedding ,只需要输入字符编号 h->1 ,e->0.....
# 所以输入形状变为【seq_len,batch_size】 ->[5,1]
# 后面的embedding层会自动把每一个编号转换成一个向量

# todo
# 为什么inputs必须是torch.long
# embedding 接收的不是普通浮点特征
# 而是要查询embedding表中第几行的整数编号
# 这里大写的Tensor不接受这种dtype参数组合
inputs = torch.tensor(x_data,dtype = torch.long).view(seq_len,batch_size)

# 将目标字符编号转化为tensor
labels = torch.tensor(y_data,dtype = torch.long).view(seq_len,batch_size)

# 创建 Embedding 层
# 设置每个字符转换后的向量长度
embedding_size = 6

# 创建 Embedding 层
# Embedding 本质上是一张可以训练的查询表
# num_embeddings 表示表中有多少行
# embedding_dim 表示每一行有多少个数字
embedding = nn.Embedding(
    num_embeddings = num_class,    # 4 字符表中一共由 4个字符
    embedding_dim = embedding_size # 6 每个字符会被自动转换成长度为6的向量
)
print('Embedding 层')
print(embedding)

# 查看embedding内部的查询表
print('\n Embedding 查询表：')
print(embedding.weight)

print('\n Embedding 查询表的形状：')
print(embedding.weight.shape)

# 将字符送入Embedding
# 将字符编号转化为 Embedding 向量
# 11. 将字符编号转换成 Embedding 向量

# input原来是【5，1】 经过embedding之后应该变成 【5，1，6】
# 【seq_len,batch_size】 ->embedding[4,6] -> [seq_len,batch_size,embedding_size]

# 这里相当于依次进行embedding.weight[1],embedding.weight[0].....
# 所以这里从【5，1】 -> 【5，1，6】
embedded_inputs = embedding(inputs)

print("\nEmbedding 前的 inputs：")
print(inputs)
print("Embedding 前的形状：", inputs.shape)

print("\nEmbedding 后的 embedded_inputs：")
print(embedded_inputs)
print("Embedding 后的形状：", embedded_inputs.shape)
print("Embedding 后的数据类型：", embedded_inputs.dtype)
print("\n第一个字符编号：")
print(inputs[0, 0])

print("\n第一个字符：")
print(idx2char[inputs[0, 0].item()])

print("\n第一个字符的 Embedding 向量：")
print(embedded_inputs[0, 0])

print("\nEmbedding 查询表的第 1 行：")
print(embedding.weight[1])

print("\n二者是否相等：")
print(
    torch.allclose(
        embedded_inputs[0, 0],
        embedding.weight[1]
    )
)

# RNN 隐藏层的长度
hidden_size = 8

# RNN层数
num_layers = 1

# 建立完整模型
class RNNEmbeddingModel(nn.Module):
    def __init__(
            self,
            num_class,
            embedding_size,
            hidden_size,
            num_layers,
    ):
        super().__init__()

        # 保存后面创建隐藏状态需要的参数
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # 将字符编号转化为向量
        self.embedding = nn.Embedding(
            num_embeddings = num_class,
            embedding_dim = embedding_size,
        )

        # 处理字符向量序列
        self.rnn = nn.RNN(
            input_size = embedding_size,
            hidden_size = hidden_size,
            num_layers = num_layers,
            batch_first = False,
        )
        # 将隐藏状态转化为四个字符的类别
        # todo
        # Linear 并不只接受二维tensor
        # 它只处理最后一个维度，前面的维度原样保留
        # 相当于对每个时间步单独调用Linear
        self.fc = nn.Linear(hidden_size,num_class)

    def forward(self,x,hidden):
        # x.shape : [seq_len,batch,batch_size]
        # [5,1] ->[5,1,6]
        embedded = self.embedding(x)

        # [5,1,6] -> [5,1,8]
        rnn_outputs,final_hidden = self.rnn(embedded,hidden)

        #[5,1,8] -> [5,1,4]
        logits = self.fc(rnn_outputs)

        return logits,final_hidden

    def init_hidden(self,batch_size):

        # nn.RNN 初始化隐藏层形状：
        # 【num+layers,batch_size,hidden_size】
        hidden = torch.zeros(
            self.num_layers,
            batch_size,
            hidden_size
        )
        return hidden

# 创建模型
rnn_net = RNNEmbeddingModel(
    num_class = num_class,
    embedding_size = embedding_size,
    hidden_size = hidden_size,
    num_layers = num_layers,
)
print(rnn_net)

# 定义损失函数和优化器
criterion = nn.CrossEntropyLoss()
# 这里的rnn_net.parameters包含完整模型的全部参数
# Embedding的查询表，RNN的权重和偏置，Linear的权重和偏置
optimizer = optim.Adam(rnn_net.parameters(),lr=0.05)

# 训练轮数
num_epochs = 15
for epoch in range(num_epochs):

    # 设置为训练模式
    rnn_net.train()
    optimizer.zero_grad()
    hidden = rnn_net.init_hidden(batch_size)

    # 向前传播
    # inputs [5,1]
    # logits [5,1,4]
    logits,final_hidden = rnn_net(inputs,hidden)

    # 调整预测结果形状
    # [5,1,4] -> [5x1,4]
    logits_2d = logits.reshape(-1,num_class)

    # 调整标签的形状
    # [5,1] ->[5]
    labels_1d = labels.reshape(-1)
    # 计算五个时间步的平均损失
    loss = criterion(logits_2d,labels_1d)

    # 反向传播，计算梯度
    loss.backward()

    # 更新模型参数
    optimizer.step()

    # 找出每个时间步的最大类别
    predicted_indices = logits_2d.argmax(dim = 1)

    # tensor 转化为 python列表
    predicted_indices = predicted_indices.tolist()

    # 类别编号转化为字符
    predicted_string = ''.join(idx2char[index] for index in predicted_indices)

    print(f'Epoch [{epoch+1} / {num_epochs}]'
          f'预测：{predicted_string}',
          f'平均损失：{loss.item(): 6f}')



Embedding 层
Embedding(4, 6)

 Embedding 查询表：
Parameter containing:
tensor([[-9.6214e-03,  1.0312e+00,  9.4613e-01, -7.8072e-01, -1.5942e-01,
          3.3073e-01],
        [-2.2187e+00,  4.2433e-02,  1.0457e+00,  1.1248e+00,  2.0944e+00,
         -4.8287e-01],
        [ 9.2218e-01, -1.2975e+00,  2.9829e-01, -1.6126e-03, -1.1638e+00,
         -1.9815e+00],
        [-3.2445e-01,  1.0082e+00, -4.1819e-02,  1.1250e+00, -6.4972e-01,
          1.3407e-01]], requires_grad=True)

 Embedding 查询表的形状：
torch.Size([4, 6])

Embedding 前的 inputs：
tensor([[1],
        [0],
        [2],
        [2],
        [3]])
Embedding 前的形状： torch.Size([5, 1])

Embedding 后的 embedded_inputs：
tensor([[[-2.2187e+00,  4.2433e-02,  1.0457e+00,  1.1248e+00,  2.0944e+00,
          -4.8287e-01]],

        [[-9.6214e-03,  1.0312e+00,  9.4613e-01, -7.8072e-01, -1.5942e-01,
           3.3073e-01]],

        [[ 9.2218e-01, -1.2975e+00,  2.9829e-01, -1.6126e-03, -1.1638e+00,
          -1.9815e+00]],

        [[ 9.2218e-01, -1.29